In [ ]:
import sys
import os
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.table import Table
import lcdata
import parsnip

# CONFIGURATION 
# Define paths using environment variables (with defaults for local usage)

ZTFIDRPATH = os.getenv("ZTFIDRPATH", "./data/ztfcosmoidr/dr2/")
ZTFDATA = os.getenv("ZTFDATA", "./data/")
PARSNIP_DIR = os.getenv("PARSNIP_DIR", "./parsnip")

os.environ['PARSNIP_DIR'] = PARSNIP_DIR

print(f"Working Directory: {os.getcwd()}")
print(f"Using PARSNIP_DIR: {PARSNIP_DIR}")

# Create directories if they don't exist
os.makedirs(os.path.join(PARSNIP_DIR, 'figures'), exist_ok=True)
os.makedirs(os.path.join(PARSNIP_DIR, 'data'), exist_ok=True)
os.makedirs(os.path.join(PARSNIP_DIR, 'predictions'), exist_ok=True)

In [ ]:
filename = os.path.join(PARSNIP_DIR, 'data', 'ps1.h5')

# Check if file exists 
if not os.path.exists(filename):
    print(f"Error: Could not find data file at {filename}")
    print("Please place 'ps1.h5' in the 'data' folder.")
else:
    # Load the dataset
    dataset_ps1 = lcdata.read_hdf5(filename, in_memory=False)
    
    # Select the first N supernovae
    N = 2000
    subset_ps1 = dataset_ps1[0:N]
    print(f"Loaded {len(subset_ps1)} light curves.")

In [ ]:
# Save the smaller subset to a new H5 file
subset_filename = os.path.join(PARSNIP_DIR, 'data', 'ps1_small_subset.h5')
subset_ps1.write_hdf5(subset_filename, overwrite=True, append=False)
print(f"Subset saved to: {subset_filename}")

In [ ]:
# Generate predictions using the pre-trained ParSNIP model
# We use the python variable {PARSNIP_DIR} to ensure it points to the right place
!parsnip_predict "{PARSNIP_DIR}/predictions/parsnip_predictions_ps1_aug_100_subset.h5" \
    "{PARSNIP_DIR}/models/parsnip_ps1.pt" \
    "{PARSNIP_DIR}/data/ps1_small_subset.h5" \
    --augments 100

In [ ]:
# Load the predictions generated above
pred_file = os.path.join(PARSNIP_DIR, 'predictions', 'parsnip_predictions_ps1_aug_100_subset.h5')
raw_predictions_ps1 = Table.read(pred_file)

# Copy predictions for analysis
predictions_ps1 = raw_predictions_ps1.copy()

In [ ]:
# Plot confusion matrix
classifier_ps1 = parsnip.Classifier()
classifications_ps1 = classifier_ps1.train(predictions_ps1, target_label='SNIa')

parsnip.plot_confusion_matrix(predictions_ps1, classifications_ps1, title='PS1 - ParSNIP')

# Save figure dynamically
fig_path = os.path.join(PARSNIP_DIR, 'figures', 'ps1_confusion_matrix_small_subset.pdf')
plt.savefig(fig_path)
print(f"Figure saved to {fig_path}")